In [1]:
from pathlib import Path
import sys
import json
import ast
import gzip
import random
import re
import hashlib
import inspect
import time
from collections import Counter
from traceback import format_exception_only
from typing import Any, Mapping

import pandas as pd
from tqdm.auto import tqdm

In [2]:
PROJECT_ROOT = Path.cwd()

RAW_VRDU_ROOT = PROJECT_ROOT / "data" / "raw" / "vrdu"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "vrdu"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_VRDU_ROOT:", RAW_VRDU_ROOT)
print("PROCESSED_ROOT:", PROCESSED_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())
print("raw vrdu exists:", RAW_VRDU_ROOT.exists())

assert (PROJECT_ROOT / "src").exists(), "Run this notebook from the repository root."
assert RAW_VRDU_ROOT.exists(), "Expected data/raw/vrdu/"

PROJECT_ROOT: /home/vios/PycharmProjects/serialization-strategies
RAW_VRDU_ROOT: /home/vios/PycharmProjects/serialization-strategies/data/raw/vrdu
PROCESSED_ROOT: /home/vios/PycharmProjects/serialization-strategies/data/processed/vrdu
src exists: True
raw vrdu exists: True


In [4]:
from src.preprocessing.schema import (
    Annotation,
    CanonicalDocument,
    OCRBlock,
    OCRToken,
)

from src.preprocessing.quality import (
    assess_document_quality,
    filter_annotations_for_training,
)

from src.preprocessing.alignment import (
    add_token_labels,
    bio_to_spans,
)

from src.serialization import (
    IGNORE_LABEL,
    PlainTextSerializer,
    PageAwareSerializer,
    BlockAwareSerializer,
    LineAwareSerializer,
    RowColBucketSerializer,
    BBoxTokenSerializer,
    ColumnAwareSerializer,
    XYCutAwareSerializer,
    LMDXCoordSuffixSerializer,
    CompactBBoxTokenSerializer,
    collapse_serialized_labels_to_original,
)

print("Imports OK")

Imports OK


## 3. Configuration

In [5]:
RAW_FILES = {
    "ad_buy_form": RAW_VRDU_ROOT / "ad_buy_form.jsonl",
    "registration_form": RAW_VRDU_ROOT / "registration_form.jsonl",
}

DATASET_NAMES = {
    "ad_buy_form": "vrdu_ad_buy_form",
    "registration_form": "vrdu_registration_form",
}

SPLIT_SEED = 42
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9

# Use None for the full dataset.
MAX_DOCS_PER_CORPUS = None

# Output controls.
WRITE_SERIALIZED = True

# Serializer controls.
INCLUDE_HEAVY_SERIALIZERS = True
HEAVY_SERIALIZERS = {}

# Alignment controls.
MIN_TOKEN_OVERLAP = 0.5
DROP_UNAVAILABLE = True
DROP_LOOSE_MISMATCH = False

# Diagnostics controls.
VALIDATE_FIRST_N_DOCS_PER_CORPUS = 10
ASSESS_QUALITY_FIRST_N_DOCS_PER_CORPUS = 25

print("RAW_FILES:")
for k, v in RAW_FILES.items():
    print(f"  {k}: {v} exists={v.exists()}")

assert all(p.exists() for p in RAW_FILES.values()), "Missing one or more raw VRDU JSONL files."

RAW_FILES:
  ad_buy_form: /home/vios/PycharmProjects/serialization-strategies/data/raw/vrdu/ad_buy_form.jsonl exists=True
  registration_form: /home/vios/PycharmProjects/serialization-strategies/data/raw/vrdu/registration_form.jsonl exists=True


In [6]:
def iter_jsonl(path: Path):
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if line:
                yield line_number, json.loads(line)

def make_with_supported_kwargs(cls, **kwargs):
    """Construct dataclass/slots classes while tolerating local schema differences."""
    params = inspect.signature(cls).parameters
    usable = {k: v for k, v in kwargs.items() if k in params}
    return cls(**usable)

def stable_seed(seed: int, name: str) -> int:
    digest = hashlib.md5(f"{seed}:{name}".encode("utf-8")).hexdigest()
    return int(digest[:8], 16)

def seeded_split(ids: list[str], corpus: str) -> dict[str, str]:
    ids = sorted(set(ids))
    rng = random.Random(stable_seed(SPLIT_SEED, corpus))
    rng.shuffle(ids)

    n = len(ids)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)

    out = {}
    for x in ids[:n_train]:
        out[x] = "train"
    for x in ids[n_train:n_train + n_val]:
        out[x] = "val"
    for x in ids[n_train + n_val:]:
        out[x] = "test"

    return out

def short_error(e: Exception) -> str:
    return "".join(format_exception_only(type(e), e)).strip()

In [7]:
raw_records = {}
load_rows = []

for corpus, path in RAW_FILES.items():
    rows = []
    for line_number, rec in iter_jsonl(path):
        rec = dict(rec)
        rec["_line_number"] = line_number
        rec["_corpus"] = corpus
        rows.append(rec)

    if MAX_DOCS_PER_CORPUS is not None:
        rows = rows[:MAX_DOCS_PER_CORPUS]

    raw_records[corpus] = rows
    load_rows.append({
        "corpus": corpus,
        "dataset": DATASET_NAMES[corpus],
        "path": str(path),
        "n_records": len(rows),
        "first_keys": list(rows[0].keys()) if rows else [],
    })

load_df = pd.DataFrame(load_rows)
display(load_df)

for corpus, rows in raw_records.items():
    print("\nCORPUS:", corpus)
    if rows:
        print("keys:", list(rows[0].keys()))
        print("filename:", rows[0].get("filename"))
        print("ocr type:", type(rows[0].get("ocr")).__name__)
        print("annotations type:", type(rows[0].get("annotations")).__name__)

,corpus,dataset,path,n_records,first_keys
0,ad_buy_form,vrdu_ad_buy_form,/home/vios/PycharmProjects/serialization-strat...,641,"[filename, file_path, ocr, annotations, _line_..."
1,registration_form,vrdu_registration_form,/home/vios/PycharmProjects/serialization-strat...,1915,"[filename, file_path, ocr, annotations, _line_..."



CORPUS: ad_buy_form
keys: ['filename', 'file_path', 'ocr', 'annotations', '_line_number', '_corpus']
filename: 00a83bbc-0101-f092-4bd7-e75f315e8f14.pdf
ocr type: dict
annotations type: list

CORPUS: registration_form
keys: ['filename', 'file_path', 'ocr', 'annotations', '_line_number', '_corpus']
filename: 19410222_DLA Piper US LLP_Amendment_Amendment.pdf
ocr type: dict
annotations type: list


In [8]:
def is_missing_value(x) -> bool:
    if x is None:
        return True
    try:
        return bool(pd.isna(x))
    except Exception:
        return False


def coerce_ocr(raw):
    """
    Handles OCR payload as:
      - dict
      - JSON string
      - Python repr string
      - None

    Your current rows have OCR in uppercase "OCR"; lowercase "ocr" is None.
    """
    if raw is None:
        return None

    if isinstance(raw, Mapping):
        return dict(raw)

    if isinstance(raw, str):
        s = raw.strip()
        if not s or s == "None":
            return None

        try:
            return json.loads(s)
        except Exception:
            pass

        try:
            return ast.literal_eval(s)
        except Exception:
            return None

    return raw


def get_ocr_payload_from_record(rec_or_row):
    """
    Prefer uppercase OCR because the sample stores the raw OCR JSON there.
    """
    upper = rec_or_row.get("OCR", None)
    lower = rec_or_row.get("ocr", None)

    if not is_missing_value(upper):
        return coerce_ocr(upper)

    if not is_missing_value(lower):
        return coerce_ocr(lower)

    return None


def get_pages(ocr):
    ocr = coerce_ocr(ocr)

    if not isinstance(ocr, Mapping):
        return []

    pages = ocr.get("pages")
    if isinstance(pages, list):
        return [p for p in pages if isinstance(p, Mapping)]

    return []


def get_page_num(obj, default=0):
    for key in ["page_id", "page_num", "page", "pageNumber", "page_index"]:
        if isinstance(obj, Mapping) and obj.get(key) is not None:
            try:
                return int(obj[key])
            except Exception:
                pass
    return default


def get_page_size(page):
    dim = page.get("dimension") or page.get("dimensions") or page.get("size") or {}

    width = dim.get("width") if isinstance(dim, Mapping) else None
    height = dim.get("height") if isinstance(dim, Mapping) else None

    width = width or page.get("width") or page.get("page_width")
    height = height or page.get("height") or page.get("page_height")

    return (
        None if width is None else int(width),
        None if height is None else int(height),
    )


def get_segments(obj):
    """
    Current VRDU OCR uses:
      segments: [[start, end]]

    Pages may use:
      segment: [start, end]
    """
    segs = obj.get("segments")

    if isinstance(segs, list) and segs:
        first = segs[0]
        if isinstance(first, (list, tuple)) and len(first) == 2:
            return int(first[0]), int(first[1])

    segment = obj.get("segment")
    if isinstance(segment, (list, tuple)) and len(segment) == 2:
        return int(segment[0]), int(segment[1])

    off = obj.get("doc_offset") or obj.get("text_offset") or obj.get("offset")
    if isinstance(off, Mapping) and off.get("start") is not None and off.get("end") is not None:
        return int(off["start"]), int(off["end"])

    return None, None


def get_bbox(obj, page_width=None, page_height=None):
    """
    Current VRDU OCR uses:
      bbox = [page, x0, y0, x1, y1]

    The x/y values are normalized floats and should be scaled by page width/height.
    """
    bbox = (
        obj.get("bbox")
        or obj.get("position")
        or obj.get("bounding_box")
        or obj.get("boundingBox")
    )

    if bbox is None:
        return None

    if isinstance(bbox, Mapping):
        left = bbox.get("left", bbox.get("x0", bbox.get("x")))
        top = bbox.get("top", bbox.get("y0", bbox.get("y")))
        right = bbox.get("right", bbox.get("x1"))
        bottom = bbox.get("bottom", bbox.get("y1"))

        if right is None and left is not None and bbox.get("width") is not None:
            right = float(left) + float(bbox["width"])

        if bottom is None and top is not None and bbox.get("height") is not None:
            bottom = float(top) + float(bbox["height"])

        vals = [left, top, right, bottom]

    elif isinstance(bbox, (list, tuple)):
        vals = list(bbox)

        # VRDU bbox format: [page, x0, y0, x1, y1]
        if len(vals) == 5:
            vals = vals[1:]

        vals = vals[:4]

    else:
        return None

    if len(vals) != 4 or any(v is None for v in vals):
        return None

    vals = [float(v) for v in vals]

    if all(0.0 <= v <= 1.0 for v in vals):
        if page_width and page_height:
            vals = [
                vals[0] * page_width,
                vals[1] * page_height,
                vals[2] * page_width,
                vals[3] * page_height,
            ]
        else:
            vals = [
                vals[0] * 1000,
                vals[1] * 1000,
                vals[2] * 1000,
                vals[3] * 1000,
            ]

    return [int(round(v)) for v in vals]


def extract_text(ocr):
    ocr = coerce_ocr(ocr)

    if not isinstance(ocr, Mapping):
        return ""

    if isinstance(ocr.get("text"), str):
        return ocr["text"]

    chunks = []
    for page in get_pages(ocr):
        if isinstance(page.get("text"), str):
            chunks.append(page["text"])

    return "\n\n".join(chunks)

In [9]:
def ocr_tokens(ocr):
    ocr = coerce_ocr(ocr)

    if not isinstance(ocr, Mapping):
        return []

    tokens = []

    for page_index, page in enumerate(get_pages(ocr)):
        page_num = get_page_num(page, page_index)
        page_width, page_height = get_page_size(page)

        for raw in page.get("tokens", []) or []:
            if not isinstance(raw, Mapping):
                continue

            text = str(raw.get("text") or "")
            start, end = get_segments(raw)
            bbox = get_bbox(raw, page_width, page_height)

            if not text or start is None or end is None or bbox is None:
                continue

            tokens.append(
                make_with_supported_kwargs(
                    OCRToken,
                    text=text,
                    start=start,
                    end=end,
                    page=page_num,
                    bbox=bbox,
                    page_width=page_width,
                    page_height=page_height,
                    style={},
                    extra={
                        "orientation": raw.get("orientation"),
                    },
                )
            )

    return sorted(
        tokens,
        key=lambda t: (
            getattr(t, "page", 0),
            getattr(t, "start", 0),
            getattr(t, "end", 0),
        ),
    )


def ocr_blocks(ocr):
    ocr = coerce_ocr(ocr)

    if not isinstance(ocr, Mapping):
        return []

    blocks = []

    for page_index, page in enumerate(get_pages(ocr)):
        page_num = get_page_num(page, page_index)
        page_width, page_height = get_page_size(page)

        block_id = 0

        # Lines are preferred as layout blocks for this OCR.
        for key in ["lines", "paragraphs"]:
            for raw in page.get(key, []) or []:
                if not isinstance(raw, Mapping):
                    continue

                text = str(raw.get("text") or "")
                start, end = get_segments(raw)
                bbox = get_bbox(raw, page_width, page_height)

                if not text or start is None or end is None or bbox is None:
                    continue

                blocks.append(
                    make_with_supported_kwargs(
                        OCRBlock,
                        text=text,
                        start=start,
                        end=end,
                        page=page_num,
                        bbox=bbox,
                        block_id=block_id,
                        block_type=key[:-1],
                        extra={
                            "orientation": raw.get("orientation"),
                        },
                    )
                )

                block_id += 1

    return sorted(
        blocks,
        key=lambda b: (
            getattr(b, "page", 0),
            getattr(b, "start", 0),
            getattr(b, "end", 0),
        ),
    )

## 6.1 OCR extraction diagnostic

In [10]:
# Quick OCR schema and token extraction diagnostic.
ocr_diag_rows = []

for corpus, rows in raw_records.items():
    for rec in rows[:5]:
        ocr = get_ocr_payload_from_record(rec)
        text = extract_text(ocr)
        toks = ocr_tokens(ocr)
        blocks = ocr_blocks(ocr)

        ocr_diag_rows.append({
            "corpus": corpus,
            "filename": rec.get("filename") or rec.get("original_filename"),
            "ocr_type": type(ocr).__name__,
            "payload_keys": list(ocr.keys()) if isinstance(ocr, Mapping) else None,
            "text_len": len(text or ""),
            "n_tokens": len(toks),
            "n_blocks": len(blocks),
            "first_20_tokens": [getattr(t, "text", None) for t in toks[:20]],
        })

ocr_diag_df = pd.DataFrame(ocr_diag_rows)
display(ocr_diag_df)

assert int(ocr_diag_df["n_tokens"].sum()) > 0, (
    "OCR parser still extracted zero tokens from the first records. "
    "Inspect payload_keys and one raw OCR payload."
)

,corpus,filename,ocr_type,payload_keys,text_len,n_tokens,n_blocks,first_20_tokens
0,ad_buy_form,00a83bbc-0101-f092-4bd7-e75f315e8f14.pdf,dict,"[text, pages]",4525,864,496,"[Page , 1 , of , 2\n, INVOICE\n, FOX\n, Remit ..."
1,ad_buy_form,00c29ad8-c88b-b3bb-1a39-3267e47c7a88.pdf,dict,"[text, pages]",1261,229,165,"[Page , 1 , of , 1\n, INVOICE\n, DUE , ATE\n, ..."
2,ad_buy_form,00c3353e-a25f-574a-a9db-39a41579895a.pdf,dict,"[text, pages]",1507,314,203,"[Print , Date , 02/28/20 , 14:21:20\n, Page , ..."
3,ad_buy_form,01250d60-2a0a-0c93-5a18-9f1f195fdc1a.pdf,dict,"[text, pages]",6724,1349,933,"[Page , 1 , of , 4\n, INVOICE\n, Advertiser\n,..."
4,ad_buy_form,0179ed96-6b64-eb8b-07c1-3214262dec0c.pdf,dict,"[text, pages]",2349,618,337,"[Contract , #\n, Date , Entered\n, Schedule , ..."
5,registration_form,19410222_DLA Piper US LLP_Amendment_Amendment.pdf,dict,"[text, pages]",4507,828,86,"[i\n, OMB , NO, . , 1124-0003\n, U.S. , Depart..."
6,registration_form,"19620326_Austrian Tourist Office, Inc._Amendme...",dict,"[text, pages]",2850,537,174,"[Budget , Bureau , No. , 42 , 9226.3\n, proval..."
7,registration_form,19630401_Arab Information Center_Amendment_Ame...,dict,"[text, pages]",4200,849,262,"[App\n, Budge , Bureau , No. , 226.3\n, Expire..."
8,registration_form,19630601_Arab Information Center_Amendment_Ame...,dict,"[text, pages]",3538,660,142,"[Feet , Bureau , No. , 43, -, P226.3\n, val , ..."
9,registration_form,19630701_KOTRA_Amendment_Amendment.pdf,dict,"[text, pages]",2941,557,206,"[Budget , Burd , Jo, . , 43, -, R227.1\n, Appr..."


## 7. Annotation parsing and `ad_buy_form` label remapping


In [11]:
PRETTY_LABELS = {
    "registration_form": {
        "file_date": "File Date",
        "foreign_principle_name": "Foreign Principal Name",
        "foreign_principal_name": "Foreign Principal Name",
        "registrant_name": "Registrant Name",
        "registration_num": "Registration Number",
        "signer_name": "Signer Name",
        "signer_title": "Signer Title",
    },
}

# ad_buy_form must use one atomic entity class per annotation/token.
# The pre-existing `labels` column in some exports contains artificial
# composites such as `channel_program_desc_sub_amount`. Those labels are
# not valid token-classification targets and are rebuilt from the original
# VRDU annotation groups whenever they are available.
AD_BUY_FORM_CANONICAL_FIELDS = (
    "advertiser",
    "agency",
    "channel",
    "contract_number",
    "flight_from",
    "flight_to",
    "gross_amount",
    "product",
    "program_desc",
    "program_start_date",
    "program_end_date",
    "property",
    "sub_amount",
    "tv_address",
)

AD_BUY_FORM_LABEL_ALIASES = {
    "advertiser": "advertiser",
    "agency": "agency",
    "channel": "channel",
    "contract_num": "contract_number",
    "contract_no": "contract_number",
    "contract_number": "contract_number",
    "flight_from": "flight_from",
    "flight_to": "flight_to",
    "gross_amount": "gross_amount",
    "product": "product",
    "program_desc": "program_desc",
    "program_description": "program_desc",
    "program_start_date": "program_start_date",
    "program_end_date": "program_end_date",
    "property": "property",
    "sub_amount": "sub_amount",
    "tv_address": "tv_address",
}

_AD_BUY_FIELDS_BY_LENGTH = sorted(
    AD_BUY_FORM_CANONICAL_FIELDS,
    key=len,
    reverse=True,
)

_AD_BUY_AMOUNT_RE = re.compile(
    r"^\s*(?:[$€£]\s*)?[-+]?\(?\d{1,3}(?:,\d{3})*(?:\.\d{2})\)?\s*$"
)
_AD_BUY_DATE_RE = re.compile(
    r"^\s*(?:"
    r"\d{1,2}[./-]\d{1,2}[./-](?:\d{2}|\d{4})"
    r"|(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|jun(?:e)?|"
    r"jul(?:y)?|aug(?:ust)?|sep(?:tember)?|oct(?:ober)?|nov(?:ember)?|"
    r"dec(?:ember)?)\s+\d{1,2}(?:,\s*|\s+)\d{2,4}"
    r")\s*$",
    re.IGNORECASE,
)


def pretty_label(corpus: str, label: str) -> str:
    label = str(label).strip()
    return PRETTY_LABELS.get(corpus, {}).get(label, label.replace("_", " ").title())


def normalize_label_key(label: Any) -> str:
    """Normalize source label spelling without changing its semantics."""
    value = str(label or "").strip()

    if value.startswith(("B-", "I-")):
        value = value[2:]

    value = value.lower()
    value = re.sub(r"[\s/.-]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value


def parse_ad_buy_form_label_components(label: Any) -> list[str]:
    """Parse an atomic/composite ad_buy_form label using longest field matches."""
    entity = normalize_label_key(label)

    direct = AD_BUY_FORM_LABEL_ALIASES.get(entity)
    if direct is not None:
        return [direct]

    remaining = entity
    components: list[str] = []

    while remaining:
        matched = None
        for field_name in _AD_BUY_FIELDS_BY_LENGTH:
            if remaining == field_name or remaining.startswith(field_name + "_"):
                matched = field_name
                break

        if matched is None:
            return []

        components.append(matched)
        remaining = remaining[len(matched):].lstrip("_")

    # Preserve order for diagnostics but remove accidental duplicates.
    return list(dict.fromkeys(components))


def infer_ad_buy_form_composite_label(
    components: list[str],
    value_text: Any,
) -> str | None:
    """Conservatively resolve only composites whose value makes one field clear.

    Most data are resolved losslessly by rebuilding annotations from the raw VRDU
    annotation groups. This fallback exists for records that only contain the
    contaminated `labels` column. Ambiguous channel/description and start/end
    date composites intentionally return None rather than inventing a class.
    """
    if len(components) == 1:
        return components[0]

    text = str(value_text or "").strip()

    if "sub_amount" in components and _AD_BUY_AMOUNT_RE.fullmatch(text):
        return "sub_amount"

    date_components = [
        field_name
        for field_name in ("program_start_date", "program_end_date")
        if field_name in components
    ]
    if len(date_components) == 1 and _AD_BUY_DATE_RE.fullmatch(text):
        return date_components[0]

    return None


def normalize_ad_buy_form_label(
    label: Any,
    value_text: Any = None,
) -> str | None:
    """Return one canonical ad_buy_form field or None when resolution is unsafe."""
    components = parse_ad_buy_form_label_components(label)
    if not components:
        return None
    return infer_ad_buy_form_composite_label(components, value_text)


def normalize_annotation_label(
    corpus: str,
    raw_label: Any,
    value_text: Any = None,
) -> str | None:
    if corpus == "ad_buy_form":
        return normalize_ad_buy_form_label(raw_label, value_text)
    return pretty_label(corpus, str(raw_label))


def is_span_pair(x: Any) -> bool:
    if not isinstance(x, (list, tuple)) or len(x) != 2:
        return False
    try:
        int(x[0])
        int(x[1])
        return True
    except Exception:
        return False


def is_vrdu_value(x: Any) -> bool:
    # Standard VRDU value:
    # ["3712\n", [0, 0.46, 0.32, 0.50, 0.34], [[2380, 2385]]]
    return (
        isinstance(x, (list, tuple))
        and len(x) >= 3
        and isinstance(x[0], str)
        and isinstance(x[2], list)
        and all(is_span_pair(s) for s in x[2])
    )


def ann_text(doc_text: str, start: int, end: int, fallback: str) -> str:
    if 0 <= start <= end <= len(doc_text):
        sliced = doc_text[start:end]
        if sliced:
            return sliced
    return fallback


def flatten_vrdu_values(x: Any) -> list[Any]:
    if is_vrdu_value(x):
        return [x]
    if isinstance(x, list):
        out = []
        for item in x:
            out.extend(flatten_vrdu_values(item))
        return out
    if isinstance(x, Mapping):
        out = []
        for item in x.values():
            out.extend(flatten_vrdu_values(item))
        return out
    return []


def parse_annotations(corpus: str, raw_annotations: Any, doc_text: str) -> list[Annotation]:
    annotations = []

    if not isinstance(raw_annotations, list):
        return annotations

    for field_item in raw_annotations:
        if not (isinstance(field_item, (list, tuple)) and len(field_item) >= 2):
            continue

        raw_label = str(field_item[0])
        values = flatten_vrdu_values(field_item[1])

        for value_index, value in enumerate(values):
            value_text = value[0]
            bbox = value[1]
            spans = value[2]
            label = normalize_annotation_label(corpus, raw_label, value_text)

            # For ad_buy_form, unknown or ambiguous labels are not converted into
            # artificial token classes. They are reported by later diagnostics.
            if label is None:
                continue

            for span_index, span in enumerate(spans):
                start, end = int(span[0]), int(span[1])
                annotations.append(
                    make_with_supported_kwargs(
                        Annotation,
                        label=label,
                        raw_label=raw_label,
                        start=start,
                        end=end,
                        text=ann_text(doc_text, start, end, value_text),
                        extra={
                            "vrdu_value_text": value_text,
                            "vrdu_bbox": bbox,
                            "vrdu_spans": spans,
                            "value_index": value_index,
                            "span_index": span_index,
                        },
                    )
                )

    return sorted(
        annotations,
        key=lambda a: (
            getattr(a, "start", 0),
            getattr(a, "end", 0),
            getattr(a, "label", ""),
        ),
    )


In [12]:
def parse_labels_column(value) -> list[dict]:
    if is_missing_value(value):
        return []

    if isinstance(value, list):
        rows = value
    elif isinstance(value, str):
        s = value.strip()
        if not s:
            return []

        try:
            rows = json.loads(s)
        except Exception:
            try:
                rows = ast.literal_eval(s)
            except Exception:
                return []
    else:
        return []

    out = []
    for x in rows:
        if not isinstance(x, Mapping):
            continue

        if x.get("start") is None or x.get("end") is None:
            continue

        out.append({
            "label": str(x.get("label")),
            "start": int(x.get("start")),
            "end": int(x.get("end")),
            "text": "" if x.get("text") is None else str(x.get("text")),
        })

    return out


def annotation_to_label_dict(a) -> dict:
    return {
        "label": str(getattr(a, "label", "")),
        "start": int(getattr(a, "start")),
        "end": int(getattr(a, "end")),
        "text": str(getattr(a, "text", "")),
    }


LABEL_PREPROCESSING_STATS = Counter()
UNRESOLVED_AD_BUY_FORM_LABELS = Counter()


def remap_existing_labels_for_corpus(
    corpus: str,
    labels: list[dict],
) -> list[dict]:
    """Normalize span-label dictionaries before CanonicalDocument creation."""
    if corpus != "ad_buy_form":
        return labels

    remapped: list[dict] = []

    for item in labels:
        raw_label = item.get("label")
        normalized = normalize_ad_buy_form_label(
            raw_label,
            item.get("text"),
        )

        if normalized is None:
            UNRESOLVED_AD_BUY_FORM_LABELS[normalize_label_key(raw_label)] += 1
            LABEL_PREPROCESSING_STATS["ad_buy_form_unresolved_spans_dropped"] += 1
            continue

        updated = dict(item)
        updated["label"] = normalized
        remapped.append(updated)

        if normalize_label_key(raw_label) == normalized:
            LABEL_PREPROCESSING_STATS["ad_buy_form_atomic_spans_kept"] += 1
        else:
            LABEL_PREPROCESSING_STATS["ad_buy_form_spans_remapped"] += 1

    return remapped


def labels_for_record(corpus: str, rec: Mapping[str, Any], doc_text: str) -> list[dict]:
    if corpus == "ad_buy_form":
        # Important: rebuild from the original VRDU field groups first. Those
        # groups retain the true atomic field identity, while some precomputed
        # `labels` columns contain merged labels generated from overlaps.
        raw_annotations = parse_annotations(
            corpus,
            rec.get("annotations"),
            doc_text,
        )
        if raw_annotations:
            LABEL_PREPROCESSING_STATS["ad_buy_form_records_rebuilt_from_annotations"] += 1
            return [annotation_to_label_dict(a) for a in raw_annotations]

        # Fallback for records without usable raw annotation groups. This path
        # only performs conservative remapping and drops unresolved composites.
        existing = parse_labels_column(rec.get("labels"))
        remapped = remap_existing_labels_for_corpus(corpus, existing)
        if remapped:
            LABEL_PREPROCESSING_STATS["ad_buy_form_records_using_labels_fallback"] += 1
        else:
            LABEL_PREPROCESSING_STATS["ad_buy_form_records_without_usable_labels"] += 1
        return remapped

    # Preserve the previous behavior for all other corpora.
    if "labels" in rec and not is_missing_value(rec.get("labels")):
        labels = parse_labels_column(rec.get("labels"))
        if labels:
            return labels

    anns = parse_annotations(corpus, rec.get("annotations"), doc_text)
    return [annotation_to_label_dict(a) for a in anns]


canonical_rows = []

for corpus, rows in raw_records.items():
    split_map = seeded_split(
        [str(r.get("filename") or r.get("original_filename") or r["_line_number"]) for r in rows],
        corpus,
    )

    for rec in tqdm(rows, desc=f"canonical {corpus}"):
        filename = str(rec.get("filename") or rec.get("original_filename") or rec["_line_number"])

        # Critical: use uppercase OCR when present.
        ocr = get_ocr_payload_from_record(rec)

        # Prefer existing nonempty text if available; otherwise extract from OCR.
        text = rec.get("text")
        if is_missing_value(text) or str(text) == "":
            text = extract_text(ocr)
        else:
            text = str(text)

        labels = labels_for_record(corpus, rec, text)

        canonical_rows.append({
            "original_filename": filename,
            "ocr": None,
            "text": text,
            "labels": json.dumps(labels, ensure_ascii=False),
            "image_files": json.dumps([], ensure_ascii=False),
            "OCR": ocr,
            "dataset": DATASET_NAMES[corpus],
            "corpus": corpus,
            "split": split_map[filename],
            "n_annotations": len(labels),
            "n_raw_annotation_groups": len(rec.get("annotations") or []),
            "line_number": rec["_line_number"],
        })

canonical_df = pd.DataFrame(canonical_rows)

display(canonical_df.head())
display(
    canonical_df.groupby(["corpus", "split"])
    .size()
    .reset_index(name="n_docs")
    .pivot(index="corpus", columns="split", values="n_docs")
    .fillna(0)
    .astype(int)
)

display(
    canonical_df.groupby("corpus")
    .agg(
        n_docs=("original_filename", "count"),
        mean_text_len=("text", lambda x: x.str.len().mean()),
        mean_annotations=("n_annotations", "mean"),
        n_without_annotations=("n_annotations", lambda x: int((x == 0).sum())),
    )
    .reset_index()
)

# Dataset-specific label diagnostics. This fails early if a composite or unknown
# ad_buy_form class survives preprocessing.
ad_buy_form_label_counts = Counter()
for value in canonical_df.loc[
    canonical_df["corpus"] == "ad_buy_form",
    "labels",
]:
    for item in parse_labels_column(value):
        ad_buy_form_label_counts[item["label"]] += 1

invalid_ad_buy_form_labels = sorted(
    set(ad_buy_form_label_counts).difference(AD_BUY_FORM_CANONICAL_FIELDS)
)

display(pd.DataFrame(
    sorted(ad_buy_form_label_counts.items()),
    columns=["label", "n_annotations"],
))
display(pd.DataFrame(
    sorted(LABEL_PREPROCESSING_STATS.items()),
    columns=["preprocessing_event", "count"],
))

if UNRESOLVED_AD_BUY_FORM_LABELS:
    display(pd.DataFrame(
        UNRESOLVED_AD_BUY_FORM_LABELS.most_common(),
        columns=["unresolved_source_label", "count"],
    ))

assert not invalid_ad_buy_form_labels, (
    "Non-canonical ad_buy_form labels survived preprocessing: "
    f"{invalid_ad_buy_form_labels}"
)


canonical ad_buy_form:   0%|          | 0/641 [00:00<?, ?it/s]

canonical registration_form:   0%|          | 0/1915 [00:00<?, ?it/s]

,original_filename,ocr,text,labels,image_files,OCR,dataset,corpus,split,n_annotations,n_raw_annotation_groups,line_number
0,00a83bbc-0101-f092-4bd7-e75f315e8f14.pdf,None,Page 1 of 2\nINVOICE\nFOX\nRemit Address:\nKMS...,"[{""label"": ""property"", ""start"": 39, ""end"": 44,...",[],{'text': 'Page 1 of 2 INVOICE FOX Remit Addres...,vrdu_ad_buy_form,ad_buy_form,test,17,16,1
1,00c29ad8-c88b-b3bb-1a39-3267e47c7a88.pdf,None,Page 1 of 1\nINVOICE\nDUE ATE\n2436316-1\nPOL/...,"[{""label"": ""advertiser"", ""start"": 38, ""end"": 5...",[],{'text': 'Page 1 of 1 INVOICE DUE ATE 2436316-...,vrdu_ad_buy_form,ad_buy_form,train,9,10,2
2,00c3353e-a25f-574a-a9db-39a41579895a.pdf,None,Print Date 02/28/20 14:21:20\nPage 1 of 1\nORD...,"[{""label"": ""contract_number"", ""start"": 70, ""en...",[],{'text': 'Print Date 02/28/20 14:21:20 Page 1 ...,vrdu_ad_buy_form,ad_buy_form,val,10,9,3
3,01250d60-2a0a-0c93-5a18-9f1f195fdc1a.pdf,None,Page 1 of 4\nINVOICE\nAdvertiser\n4\nRemit Add...,"[{""label"": ""property"", ""start"": 48, ""end"": 56,...",[],{'text': 'Page 1 of 4 INVOICE Advertiser 4 Rem...,vrdu_ad_buy_form,ad_buy_form,train,33,26,4
4,0179ed96-6b64-eb8b-07c1-3214262dec0c.pdf,None,Contract #\nDate Entered\nSchedule Dates\nLast...,"[{""label"": ""gross_amount"", ""start"": 207, ""end""...",[],{'text': 'Contract # Date Entered Schedule Dat...,vrdu_ad_buy_form,ad_buy_form,train,9,18,5


split,test,train,val
corpus,,,
ad_buy_form,65,512,64
registration_form,192,1532,191


,corpus,n_docs,mean_text_len,mean_annotations,n_without_annotations
0,ad_buy_form,641,7353.109204,16.000000,0
1,registration_form,1915,5212.368146,4.781201,0


,label,n_annotations
0,advertiser,1349
1,agency,440
2,contract_number,1200
3,flight_from,1025
4,flight_to,1021
5,gross_amount,1165
6,product,1267
7,property,1629
8,tv_address,1160


,preprocessing_event,count
0,ad_buy_form_records_rebuilt_from_annotations,641


In [13]:
sample = canonical_df.iloc[0]
sample_ocr = coerce_ocr(sample["OCR"])

sample_tokens = ocr_tokens(sample_ocr)
sample_blocks = ocr_blocks(sample_ocr)
sample_labels = json.loads(sample["labels"])

print("sample filename:", sample["original_filename"])
print("corpus:", sample["corpus"])
print("text length:", len(sample["text"]))
print("n labels:", len(sample_labels))
print("n OCR tokens:", len(sample_tokens))
print("n OCR blocks:", len(sample_blocks))

display(pd.DataFrame([
    {
        "i": i,
        "text": t.text,
        "start": t.start,
        "end": t.end,
        "page": t.page,
        "bbox": t.bbox,
    }
    for i, t in enumerate(sample_tokens[:50])
]))

display(pd.DataFrame(sample_labels[:30]))

assert len(sample["text"]) > 0, "text is still empty"
assert len(sample_tokens) > 0, "OCR tokens are still empty"
assert len(sample_labels) > 0, "labels are empty"

sample filename: 00a83bbc-0101-f092-4bd7-e75f315e8f14.pdf
corpus: ad_buy_form
text length: 4525
n labels: 17
n OCR tokens: 864
n OCR blocks: 496


,i,text,start,end,page,bbox
0,0,Page,0,5,0,"[2028, 53, 2083, 79]"
1,1,1,5,7,0,"[2109, 52, 2116, 77]"
2,2,of,7,10,0,"[2130, 52, 2153, 77]"
3,3,2\n,10,12,0,"[2164, 51, 2171, 76]"
4,4,INVOICE\n,12,20,0,"[1066, 64, 1254, 102]"
5,5,FOX\n,20,24,0,"[106, 78, 270, 144]"
6,6,Remit,24,30,0,"[346, 89, 408, 111]"
7,7,Address,30,37,0,"[416, 89, 504, 111]"
8,8,:\n,37,39,0,"[510, 89, 516, 111]"
9,9,KMSP\n,39,44,0,"[339, 122, 425, 151]"


,label,start,end,text
0,property,39,44,KMSP\n
1,tv_address,44,91,"4614 Collection Center Drive\nChicago, IL 60693\n"
2,advertiser,171,199,"Michael Bloomberg 2020, Inc\n"
3,product,199,223,MIKE BLOOMBERG 2020 INC\n
4,contract_number,378,385,950658\n
5,property,406,411,KMSP\n
6,flight_from,530,539,12/30/19
7,flight_to,541,550,03/29/20\n
8,property,2541,2546,KMSP\n
9,tv_address,2546,2593,"4614 Collection Center Drive\nChicago, IL 60693\n"


In [14]:
def row_to_document(row: pd.Series) -> CanonicalDocument:
    ocr = coerce_ocr(row["OCR"])
    text = row["text"] if isinstance(row["text"], str) and row["text"] else extract_text(ocr)
    labels = json.loads(row["labels"]) if isinstance(row["labels"], str) else row["labels"]

    annotations = [
        make_with_supported_kwargs(
            Annotation,
            label=str(x["label"]),
            raw_label=str(x["label"]),
            start=int(x["start"]),
            end=int(x["end"]),
            text=str(x.get("text") or text[int(x["start"]):int(x["end"])]),
            extra={},
        )
        for x in labels
        if x.get("start") is not None and x.get("end") is not None
    ]

    return make_with_supported_kwargs(
        CanonicalDocument,
        doc_id=str(row["original_filename"]),
        dataset_name=str(row["dataset"]),
        text=text,
        tokens=ocr_tokens(ocr),
        blocks=ocr_blocks(ocr),
        annotations=annotations,
        metadata={
            "corpus": row["corpus"],
            "split": row["split"],
            "line_number": int(row["line_number"]),
        },
    )

docs = []
doc_errors = []

for _, row in tqdm(canonical_df.iterrows(), total=len(canonical_df), desc="CanonicalDocument"):
    try:
        doc = row_to_document(row)
        docs.append(doc)
    except Exception as e:
        doc_errors.append({
            "corpus": row.get("corpus"),
            "filename": row.get("original_filename"),
            "error": short_error(e),
        })

doc_diag_df = pd.DataFrame([
    {
        "corpus": d.metadata.get("corpus"),
        "split": d.metadata.get("split"),
        "doc_id": d.doc_id,
        "n_tokens": len(d.tokens),
        "n_blocks": len(d.blocks or []),
        "n_annotations": len(d.annotations or []),
        "text_len": len(d.text or ""),
    }
    for d in docs
])

doc_errors_df = pd.DataFrame(doc_errors)

display(doc_diag_df.head())
display(doc_errors_df.head(50))

display(
    doc_diag_df.groupby("corpus")
    .agg(
        n_docs=("doc_id", "count"),
        mean_tokens=("n_tokens", "mean"),
        mean_blocks=("n_blocks", "mean"),
        mean_annotations=("n_annotations", "mean"),
        n_zero_token_docs=("n_tokens", lambda x: int((x == 0).sum())),
        n_zero_annotation_docs=("n_annotations", lambda x: int((x == 0).sum())),
    )
    .reset_index()
)

assert len(docs) > 0, "No CanonicalDocument objects created."
assert int((doc_diag_df["n_tokens"] == 0).sum()) == 0, "Some docs have zero tokens. Inspect doc_diag_df."

CanonicalDocument:   0%|          | 0/2556 [00:00<?, ?it/s]

,corpus,split,doc_id,n_tokens,n_blocks,n_annotations,text_len
0,ad_buy_form,test,00a83bbc-0101-f092-4bd7-e75f315e8f14.pdf,864,496,17,4525
1,ad_buy_form,train,00c29ad8-c88b-b3bb-1a39-3267e47c7a88.pdf,229,165,9,1261
2,ad_buy_form,val,00c3353e-a25f-574a-a9db-39a41579895a.pdf,314,203,10,1507
3,ad_buy_form,train,01250d60-2a0a-0c93-5a18-9f1f195fdc1a.pdf,1349,933,33,6724
4,ad_buy_form,train,0179ed96-6b64-eb8b-07c1-3214262dec0c.pdf,618,337,9,2349


""


,corpus,n_docs,mean_tokens,mean_blocks,mean_annotations,n_zero_token_docs,n_zero_annotation_docs
0,ad_buy_form,641,1524.419657,536.246490,16.000000,0,0
1,registration_form,1915,948.828721,134.107572,4.781201,0,0


## 10. Alignment and BIO labels

In [15]:
prepared_docs = []
alignment_rows = []

seen_by_corpus = Counter()

for doc in tqdm(docs, desc="align"):
    corpus = doc.metadata.get("corpus")
    idx = seen_by_corpus[corpus]
    seen_by_corpus[corpus] += 1

    try:
        quality = None
        if idx < ASSESS_QUALITY_FIRST_N_DOCS_PER_CORPUS:
            quality = assess_document_quality(doc, min_token_overlap=MIN_TOKEN_OVERLAP)

        filtered = filter_annotations_for_training(
            doc,
            min_token_overlap=MIN_TOKEN_OVERLAP,
            drop_unavailable=DROP_UNAVAILABLE,
            drop_loose_mismatch=DROP_LOOSE_MISMATCH,
            attach_quality_report=quality is not None,
        )

        if not filtered.annotations:
            raise ValueError("No annotations left after filtering.")

        labeled = add_token_labels(
            filtered,
            min_token_overlap=MIN_TOKEN_OVERLAP,
            conflict_policy="keep_first",
            include_alignment_diagnostics=False,
        )

        token_labels = labeled.metadata["token_labels"]

        if corpus == "ad_buy_form":
            invalid_token_labels = sorted({
                str(label)
                for label in token_labels
                if (
                    label not in {"O", IGNORE_LABEL, str(IGNORE_LABEL)}
                    and (
                        not str(label).startswith(("B-", "I-"))
                        or str(label)[2:] not in AD_BUY_FORM_CANONICAL_FIELDS
                    )
                )
            })
            if invalid_token_labels:
                raise ValueError(
                    "Non-canonical ad_buy_form token labels after alignment: "
                    f"{invalid_token_labels}"
                )

        prepared_docs.append(labeled)

        alignment_rows.append({
            "corpus": corpus,
            "split": doc.metadata.get("split"),
            "doc_id": doc.doc_id,
            "status": "ok",
            "n_tokens": len(labeled.tokens),
            "n_annotations_before": len(doc.annotations or []),
            "n_annotations_after": len(labeled.annotations or []),
            "n_non_o_token_labels": sum(1 for x in token_labels if x != "O"),
            "token_available_rate": None if quality is None else quality.token_available_rate,
            "strict_match_rate": None if quality is None else quality.strict_match_rate,
            "loose_value_match_rate": None if quality is None else quality.loose_value_match_rate,
            "error": None,
        })

    except Exception as e:
        alignment_rows.append({
            "corpus": corpus,
            "split": doc.metadata.get("split"),
            "doc_id": doc.doc_id,
            "status": "error",
            "n_tokens": len(doc.tokens),
            "n_annotations_before": len(doc.annotations or []),
            "n_annotations_after": None,
            "n_non_o_token_labels": None,
            "token_available_rate": None,
            "strict_match_rate": None,
            "loose_value_match_rate": None,
            "error": short_error(e),
        })

alignment_df = pd.DataFrame(alignment_rows)

display(alignment_df.head())
display(
    alignment_df.groupby(["corpus", "status"])
    .agg(
        n_docs=("doc_id", "count"),
        mean_tokens=("n_tokens", "mean"),
        mean_annotations_after=("n_annotations_after", "mean"),
        mean_non_o=("n_non_o_token_labels", "mean"),
        mean_token_available_rate=("token_available_rate", "mean"),
        mean_loose_value_match_rate=("loose_value_match_rate", "mean"),
    )
    .reset_index()
)
display(alignment_df[alignment_df["status"] == "error"].head(50))

print("prepared docs:", len(prepared_docs))

align:   0%|          | 0/2556 [00:00<?, ?it/s]

,corpus,split,doc_id,status,n_tokens,n_annotations_before,n_annotations_after,n_non_o_token_labels,token_available_rate,strict_match_rate,loose_value_match_rate,error
0,ad_buy_form,test,00a83bbc-0101-f092-4bd7-e75f315e8f14.pdf,ok,864,17,17,46,1.0,0.705882,1.0,None
1,ad_buy_form,train,00c29ad8-c88b-b3bb-1a39-3267e47c7a88.pdf,ok,229,9,9,19,1.0,0.666667,1.0,None
2,ad_buy_form,val,00c3353e-a25f-574a-a9db-39a41579895a.pdf,ok,314,10,10,29,1.0,0.600000,1.0,None
3,ad_buy_form,train,01250d60-2a0a-0c93-5a18-9f1f195fdc1a.pdf,ok,1349,33,33,78,1.0,0.606061,1.0,None
4,ad_buy_form,train,0179ed96-6b64-eb8b-07c1-3214262dec0c.pdf,ok,618,9,9,44,1.0,0.222222,1.0,None


,corpus,status,n_docs,mean_tokens,mean_annotations_after,mean_non_o,mean_token_available_rate,mean_loose_value_match_rate
0,ad_buy_form,ok,641,1524.419657,16.000000,51.205928,1.0,1.0
1,registration_form,ok,1915,948.828721,4.781201,16.402611,1.0,1.0


,corpus,split,doc_id,status,n_tokens,n_annotations_before,n_annotations_after,n_non_o_token_labels,token_available_rate,strict_match_rate,loose_value_match_rate,error


prepared docs: 2556


## 11. Serializers

In [16]:
all_serializers = {
    "plain_text": PlainTextSerializer(),
    "page_aware": PageAwareSerializer(),
    "block_aware": BlockAwareSerializer(),
    "line_aware": LineAwareSerializer(),
    "column_aware": ColumnAwareSerializer(
        min_gap_ratio=0.035,
        min_tokens_per_column=8,
        max_columns=4,
    ),
    "xycut_aware": XYCutAwareSerializer(
        min_gap_ratio_x=0.06,
        min_gap_ratio_y=0.035,
        min_tokens_per_region=8,
        max_depth=6,
    ),
    "lmdx_coord_suffix": LMDXCoordSuffixSerializer(
        n_buckets=100,
        coord_mode="center",
    ),
    "compact_bbox_token": CompactBBoxTokenSerializer(
        n_buckets=100,
        position="prefix",
    ),
    "rowcol_bucket": RowColBucketSerializer(n_buckets=100),
    "bbox_token": BBoxTokenSerializer(n_buckets=100),
}

serializers = {
    name: serializer
    for name, serializer in all_serializers.items()
    if INCLUDE_HEAVY_SERIALIZERS or name not in HEAVY_SERIALIZERS
}

print("serializers:", sorted(serializers))

serializers: ['bbox_token', 'block_aware', 'column_aware', 'compact_bbox_token', 'line_aware', 'lmdx_coord_suffix', 'page_aware', 'plain_text', 'rowcol_bucket', 'xycut_aware']


In [17]:
def validate_serialized_record(record: dict, doc: CanonicalDocument, token_labels: list[str]) -> None:
    n = len(record["tokens"])
    same_length_keys = [
        "source_token_indices",
        "loss_mask",
        "layout_roles",
        "item_attrs",
        "labels",
        "pages",
        "bboxes",
        "normalized_bboxes",
        "offsets",
    ]

    for key in same_length_keys:
        assert key in record, f"missing {key}"
        assert len(record[key]) == n, f"{record['serializer']} {key} length mismatch"

    real_indices = [i for i in record["source_token_indices"] if i is not None]
    assert sorted(real_indices) == list(range(len(doc.tokens)))

    for pos, idx in enumerate(record["source_token_indices"]):
        if idx is None:
            assert record["loss_mask"][pos] is False
            assert record["labels"][pos] == IGNORE_LABEL
        else:
            assert record["loss_mask"][pos] is True
            assert record["labels"][pos] == token_labels[idx]

    collapsed = collapse_serialized_labels_to_original(record, record["labels"])
    assert collapsed == token_labels

def add_serialized_metadata(record: dict, doc: CanonicalDocument) -> dict:
    record = dict(record)
    record["corpus"] = doc.metadata.get("corpus")
    record["split"] = doc.metadata.get("split")
    record["doc_id"] = doc.doc_id
    record["split_seed"] = SPLIT_SEED

    record.setdefault("metadata", {})
    if isinstance(record["metadata"], dict):
        record["metadata"]["corpus"] = doc.metadata.get("corpus")
        record["metadata"]["split"] = doc.metadata.get("split")
        record["metadata"]["split_seed"] = SPLIT_SEED

    return record

class LazyJsonlWriter:
    def __init__(self, root: Path):
        self.root = Path(root)
        self.handles = {}
        self.counts = Counter()

    def path_for(self, corpus: str, serializer: str, split: str) -> Path:
        return self.root / "serialized" / corpus / serializer / f"{split}.jsonl"

    def write(self, corpus: str, serializer: str, split: str, record: dict) -> None:
        key = (corpus, serializer, split)
        if key not in self.handles:
            path = self.path_for(corpus, serializer, split)
            path.parent.mkdir(parents=True, exist_ok=True)
            self.handles[key] = open(path, "w", encoding="utf-8")

        self.handles[key].write(json.dumps(record, ensure_ascii=False) + "\n")
        self.counts[key] += 1

    def close(self):
        for handle in self.handles.values():
            handle.flush()
            handle.close()
        self.handles = {}

    def summary(self) -> pd.DataFrame:
        rows = []
        for (corpus, serializer, split), n in sorted(self.counts.items()):
            p = self.path_for(corpus, serializer, split)
            rows.append({
                "corpus": corpus,
                "serializer": serializer,
                "split": split,
                "n_records": n,
                "path": str(p),
                "exists": p.exists(),
                "size_bytes": p.stat().st_size if p.exists() else 0,
            })
        return pd.DataFrame(rows)

## 12. Serialize

In [18]:
writer = LazyJsonlWriter(PROCESSED_ROOT) if WRITE_SERIALIZED else None

stats_rows = []
error_rows = []
preview_records = {}

for doc in tqdm(prepared_docs, desc="serialize"):
    corpus = doc.metadata.get("corpus")
    idx = seen_by_corpus[corpus]
    seen_by_corpus[corpus] += 1

    token_labels = doc.metadata["token_labels"]
    validate_this = idx < VALIDATE_FIRST_N_DOCS_PER_CORPUS

    for serializer_name, serializer in serializers.items():
        try:
            t0 = time.perf_counter()
            record = serializer.serialize_train(doc)
            seconds = time.perf_counter() - t0

            if validate_this:
                validate_serialized_record(record, doc, token_labels)

            record = add_serialized_metadata(record, doc)

            if writer is not None:
                writer.write(corpus, serializer_name, "all", record)
                writer.write(corpus, serializer_name, doc.metadata.get("split"), record)

            preview_records.setdefault((corpus, serializer_name), record)

            real = sum(record["loss_mask"])
            stats_rows.append({
                "corpus": corpus,
                "dataset": doc.dataset_name,
                "split": doc.metadata.get("split"),
                "doc_id": doc.doc_id,
                "serializer": serializer_name,
                "validated": validate_this,
                "original_token_count": len(doc.tokens),
                "serialized_token_count": len(record["tokens"]),
                "real_token_count": real,
                "layout_token_count": len(record["tokens"]) - real,
                "length_multiplier": len(record["tokens"]) / max(1, len(doc.tokens)),
                "n_annotations": len(doc.annotations or []),
                "n_non_o_token_labels": sum(1 for x in token_labels if x != "O"),
                "serialize_seconds": seconds,
            })

        except Exception as e:
            error_rows.append({
                "corpus": corpus,
                "doc_id": doc.doc_id,
                "serializer": serializer_name,
                "error": short_error(e),
            })

if writer is not None:
    writer.close()


serialize:   0%|          | 0/2556 [00:00<?, ?it/s]